# Present-day snapshot

**A point-in-time cross-section from this project's own live-collected data
— not a time series.** Companion to `notebooks/esports_share_of_twitch.ipynb`
(Kaggle-based, 2016-2024) and `notebooks/niche_stacked_area.ipynb`
(long-run per-niche composition); this notebook asks "where do things stand
right now," using only data this project has actually collected itself.

**Window used, stated explicitly and prominently**: collection start
(`2026-08-31`) through today. Computed directly below, not assumed. **Flag
this as preliminary for two independent reasons, both real**: (1) the window
itself is short — about a week as of this run; (2) polling within that
window is confirmed sparse and irregular (see
`etl/compute_monthly_category_totals.py`'s docstring — 40+ polls with gaps
from ~5 minutes to ~7.6 hours, not a clean hourly cadence). Both limit how
much confidence any figure below deserves.

In [1]:
# Imports and repo path setup — same pattern as the other notebooks in this repo.
import sys
from collections import defaultdict
from datetime import datetime
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))

import pandas as pd
import yaml

from etl.db import get_connection

conn = get_connection()

window_start, window_end = conn.execute(
    "SELECT MIN(captured_at), MAX(captured_at) FROM viewership_snapshots"
).fetchone()
print(f"Window used: {window_start} through {window_end}")
print("(collection start through the most recent poll as of this run — this is NOT necessarily "
      "the moment this notebook was written; re-running it later widens the window automatically.)")

Window used: 2026-08-31T10:13:48Z through 2026-09-07T14:13:18Z
(collection start through the most recent poll as of this run — this is NOT necessarily the moment this notebook was written; re-running it later widens the window automatically.)


## Time-weighted hours — one shared method, used consistently below

Every "hours" figure in this notebook (category-wide, esports-specific, and
whole-platform) is computed the same way: for each poll, weight its
viewer/concurrent-viewer count by the actual time (in hours) until the NEXT
poll of that same series — a real timestamp delta, not an assumed constant,
matching `etl/compute_monthly_category_totals.py`'s method exactly (reused
here, not reinvented). The very last poll of any series gets no
forward-looking weight — we cannot extrapolate into unobserved time, so the
most recent partial period is a deliberate, honest undercount everywhere
below, not a bug.

In [2]:
def time_weighted_hours(timestamped_values: list[tuple[datetime, float]]) -> float:
    """timestamped_values: list of (timestamp, value) for ONE series, any order.
    Returns sum(value * hours_until_next_timestamp), last point uncounted forward."""
    ordered = sorted(timestamped_values)
    total = 0.0
    for i in range(len(ordered) - 1):
        ts, val = ordered[i]
        next_ts, _ = ordered[i + 1]
        hours = (next_ts - ts).total_seconds() / 3600
        total += val * hours
    return total

## Metric (a): tracked-title category hours as % of whole-Twitch hours

**Numerator**: sum of `monthly_category_history`'s `source='own_collector'`
rows (all 23 tracked titles) across the window — this is a game-fandom
figure (ranked play, guides, cosmetics content included, per PRD §4's own
framing), not esports-specific.

**Denominator**: `platform_totals`, time-weighted the same way. **State
this plainly: `platform_totals` is a TRUE whole-Twitch denominator — every
category, not a top-200 cutoff.** This is the key methodological difference
from `esports_share_of_twitch.ipynb`'s Kaggle-based share, whose denominator
was the sum of the top-200 ranked categories only (anything outside that
ranking each month was excluded from both sides there). A broader
denominator here should, all else equal, produce an EQUAL OR LOWER share
than the Kaggle-style top-200-only denominator would — that mechanical
expectation matters for reading the comparison at the end of this
notebook.

In [3]:
# Numerator: own_collector category hours, all 23 tracked titles, whole window.
category_hours = conn.execute(
    "SELECT SUM(hours_watched) FROM monthly_category_history WHERE source = 'own_collector'"
).fetchone()[0]
print(f"tracked-title category hours (own_collector, whole window): {category_hours:,.0f}")

# Denominator: platform_totals, time-weighted.
pt_rows = conn.execute("SELECT captured_at, total_viewers FROM platform_totals ORDER BY captured_at").fetchall()
print(f"\nplatform_totals sample count: {len(pt_rows)}")
for captured_at, total_viewers in pt_rows:
    print(f"  {captured_at}: {total_viewers:,} concurrent viewers")
if len(pt_rows) <= 2:
    print(f"\n[CAVEAT, stated prominently] Only {len(pt_rows)} platform_totals sample(s) exist — "
          "every scheduled hourly poll defaults to --skip-platform-totals (a known gap, see "
          "docs/system_reference.md), so this denominator rests on very few points. With 2 samples, "
          "the whole-window total below is dominated by extrapolating the FIRST sample's snapshot "
          "across nearly the entire window (the last sample gets no forward weight) — read the "
          "resulting percentage as a rough order-of-magnitude estimate, not a precise figure.")

platform_series = [(datetime.strptime(ts, "%Y-%m-%dT%H:%M:%SZ"), v) for ts, v in pt_rows]
platform_hours = time_weighted_hours(platform_series)
print(f"\nwhole-Twitch hours (time-weighted, whole window): {platform_hours:,.0f}")

metric_a = 100 * category_hours / platform_hours
print(f"\nMETRIC (a): tracked-title category hours / whole-Twitch hours = {metric_a:.1f}%")

tracked-title category hours (own_collector, whole window): 78,753,863

platform_totals sample count: 2
  2026-08-31T10:13:48Z: 1,350,715 concurrent viewers
  2026-09-07T14:13:18Z: 1,876,963 concurrent viewers

[CAVEAT, stated prominently] Only 2 platform_totals sample(s) exist — every scheduled hourly poll defaults to --skip-platform-totals (a known gap, see docs/system_reference.md), so this denominator rests on very few points. With 2 samples, the whole-window total below is dominated by extrapolating the FIRST sample's snapshot across nearly the entire window (the last sample gets no forward weight) — read the resulting percentage as a rough order-of-magnitude estimate, not a precise figure.

whole-Twitch hours (time-weighted, whole window): 232,311,724

METRIC (a): tracked-title category hours / whole-Twitch hours = 33.9%


## Metric (b): esports-specific hours as % of whole-Twitch hours

**A figure Kaggle could never produce** — it has no broadcast-tier concept
at all. Numerator: `viewership_snapshots` rows where `broadcast_tier` is
`primary_official` or `detected_costream`, converted to hours with the same
time-weighted method above (reusing each title's own full poll timeline for
the "time until next poll" weight, not just the esports-tier polls in
isolation — a title's non-esports polls still mark real time boundaries).
Same `platform_totals` denominator as metric (a), so the two are directly
comparable to each other (not to Kaggle).

In [4]:
# Esports-tier rows, summed per (title_id, captured_at).
esports_rows = conn.execute(
    """
    SELECT title_id, captured_at, SUM(viewer_count)
    FROM viewership_snapshots
    WHERE broadcast_tier IN ('primary_official', 'detected_costream')
    GROUP BY title_id, captured_at
    """
).fetchall()

# Full poll timeline per title (needed for correct "hours to next poll" weights,
# not just the esports-tier polls in isolation).
all_polls = conn.execute("SELECT DISTINCT title_id, captured_at FROM viewership_snapshots").fetchall()
all_ts_by_title = defaultdict(list)
for title_id, captured_at in all_polls:
    all_ts_by_title[title_id].append(datetime.strptime(captured_at, "%Y-%m-%dT%H:%M:%SZ"))
for title_id in all_ts_by_title:
    all_ts_by_title[title_id].sort()

esports_hours_total = 0.0
for title_id, captured_at, viewers in esports_rows:
    ts = datetime.strptime(captured_at, "%Y-%m-%dT%H:%M:%SZ")
    all_ts = all_ts_by_title[title_id]
    idx = all_ts.index(ts)
    if idx + 1 < len(all_ts):
        hours_to_next = (all_ts[idx + 1] - ts).total_seconds() / 3600
        esports_hours_total += viewers * hours_to_next

print(f"esports-specific hours (broadcast_tier in official/detected_costream, whole window): "
      f"{esports_hours_total:,.0f}")

metric_b = 100 * esports_hours_total / platform_hours
print(f"\nMETRIC (b): esports-specific hours / whole-Twitch hours = {metric_b:.1f}%")

print(f"\nSide by side:")
print(f"  (a) game-fandom category share:  {metric_a:.1f}%")
print(f"  (b) esports-specific share:      {metric_b:.1f}%")
print(f"  esports is {100*esports_hours_total/category_hours:.1f}% of tracked titles' own category hours "
      f"— i.e. most of these titles' Twitch attention is NOT tournament broadcasts, consistent with "
      f"PRD §4's game-fandom-vs-esports-fandom distinction.")

esports-specific hours (broadcast_tier in official/detected_costream, whole window): 16,220,132

METRIC (b): esports-specific hours / whole-Twitch hours = 7.0%

Side by side:
  (a) game-fandom category share:  33.9%
  (b) esports-specific share:      7.0%
  esports is 20.6% of tracked titles' own category hours — i.e. most of these titles' Twitch attention is NOT tournament broadcasts, consistent with PRD §4's game-fandom-vs-esports-fandom distinction.


### Per-title esports-specific ratio — not just the 33.9%/7.0% aggregate

The aggregate metric (b) above (7.0%) blends all 23 titles together. That
hides two things worth seeing directly: metric (b) is **entirely**
`detected_costream` (the text-match heuristic in
`etl/classify_broadcast_tier.py`) — `primary_official` is 0 rows everywhere
in this window, since `config/channels.yaml` is empty and no channel has
ever been captured as official at ingest time (`is_official_broadcast=1`
is 0 rows in `viewership_snapshots`, full stop). And it's not spread evenly
across titles at all — see the concentration finding below the table.

In [5]:
# Per-title esports-specific hours (same detected_costream/primary_official filter and
# time-weighted conversion as metric (b) above), divided by that title's own category hours.
per_title_esports_hours = defaultdict(float)
for title_id, captured_at, viewers in esports_rows:
    ts = datetime.strptime(captured_at, "%Y-%m-%dT%H:%M:%SZ")
    all_ts = all_ts_by_title[title_id]
    idx = all_ts.index(ts)
    if idx + 1 < len(all_ts):
        hours_to_next = (all_ts[idx + 1] - ts).total_seconds() / 3600
        per_title_esports_hours[title_id] += viewers * hours_to_next

detected_costream_counts = dict(conn.execute(
    "SELECT title_id, COUNT(*) FROM viewership_snapshots WHERE broadcast_tier = 'detected_costream' "
    "GROUP BY title_id"
).fetchall())

all_title_ids = [t["id"] for t in yaml.safe_load(open(REPO_ROOT / "config" / "titles.yaml"))["titles"]]

ratio_rows = []
title_category_hours = dict(conn.execute(
    "SELECT title_id, SUM(hours_watched) FROM monthly_category_history WHERE source = 'own_collector' "
    "GROUP BY title_id"
).fetchall())
for t in all_title_ids:
    cat_hrs = title_category_hours.get(t, 0.0)
    esp_hrs = per_title_esports_hours.get(t, 0.0)
    ratio = 100 * esp_hrs / cat_hrs if cat_hrs else 0.0
    ratio_rows.append({
        "title_id": t,
        "esports_specific_hours": round(esp_hrs, 0),
        "category_hours": round(cat_hrs, 0),
        "ratio_pct": round(ratio, 2),
        "detected_costream_rows": detected_costream_counts.get(t, 0),
    })

ratio_df = pd.DataFrame(ratio_rows).sort_values("ratio_pct", ascending=False).reset_index(drop=True)
print(ratio_df.to_string(index=False))

         title_id  esports_specific_hours  category_hours  ratio_pct  detected_costream_rows
   counter_strike               7266355.0      14055099.0      51.70                     738
league_of_legends               5629303.0      16447577.0      34.23                     692
age_of_empires_ii                 92863.0        325894.0      28.49                      84
     apex_legends                970602.0       4325388.0      22.44                     566
   street_fighter                243915.0       1477914.0      16.50                     166
           tekken                 49067.0        309803.0      15.84                     104
         valorant               1497314.0      10716429.0      13.97                     492
rainbow_six_siege                383389.0       3049030.0      12.57                      69
        free_fire                  3589.0         28575.0      12.56                      12
mobile_legends_bb                 44649.0        420204.0      10.63  

### Concentration finding

**Only 13 of 23 tracked titles have any `detected_costream` row at all in
this window — 10 have zero**: `dota2`, `fortnite`, `hearthstone`,
`teamfight_tactics`, `wild_rift`, `pubg`, `starcraft2`, `brawl_stars`,
`guilty_gear`, `mortal_kombat`. And among the 13 that do, it's heavily
concentrated: `counter_strike` (738 rows), `league_of_legends` (692),
`apex_legends` (566), and `valorant` (492) together account for **83% of
all 2,997 `detected_costream` rows** in this window. The 7.0% aggregate
figure is not a broad, evenly-distributed signal — it's dominated by
whichever handful of titles happened to have an active tournament (or
tournament-tagged streams) during this specific week.

### Tag-match caveat — a concrete example, not a hypothetical

`detected_costream` is `confidence='proxy_estimate'` by design (see
`etl/classify_broadcast_tier.py`'s own docstring) — it's a text/tag match
against tournament aliases during an active tournament's date window, not a
verified check that genuine co-streaming is happening. A real example found
directly in this window's data: an Apex Legends stream titled **"Lets vibe
out and get this RP til i crash out"** matched `detected_costream` — not
because it was actually co-streaming ALGS, but because its `tags` field
included `algs` (among `black, africanamerican, competitive, ranked, ewc,
apex, fitness, English`) while an ALGS tournament happened to be running.
This is the schema's own documented tradeoff, not a bug — but it means
every ratio in the table above is a **tag-inclusive proxy**, not a verified
co-stream count. Read them as "how often a title's streams carried
tournament-adjacent signals," not "how much genuine co-streaming
happened."

In [6]:
# The concrete example cited above, pulled directly from this window's data (not paraphrased).
example = conn.execute(
    """
    SELECT stream_title, tags, matched_alias FROM viewership_snapshots
    WHERE stream_title LIKE '%vibe out%crash out%' LIMIT 1
    """
).fetchone()
print(f"stream_title: {example[0]!r}")
print(f"tags:         {example[1]!r}")
print(f"matched_alias: {example[2]!r}")

stream_title: 'Lets vibe out and get this RP til i crash out'
tags:         'algs,black,africanamerican,competitive,ranked,ewc,apex,fitness,English'
matched_alias: 'ALGS'


### Cross-tab against esports_share_of_twitch.ipynb's pre-wave/post-wave cohorts

Reusing the EXACT pre-wave (13 titles) / post-wave (10 titles) split from
that notebook's original comparison (first top-200 appearance before vs.
on/after October 2018 — not the restricted 6-vs-10 re-run), read directly
from its executed cells so the two notebooks stay consistent with each
other, not reconstructed independently.

In [7]:
PRE_WAVE = ["age_of_empires_ii", "brawl_stars", "counter_strike", "dota2", "fortnite", "hearthstone",
            "league_of_legends", "overwatch", "pubg", "pubg_mobile", "rainbow_six_siege",
            "rocket_league", "starcraft2"]
POST_WAVE = ["apex_legends", "free_fire", "guilty_gear", "mobile_legends_bb", "mortal_kombat",
             "street_fighter", "teamfight_tactics", "tekken", "valorant", "wild_rift"]
assert set(PRE_WAVE) | set(POST_WAVE) == set(all_title_ids), "cohort lists should cover all 23 tracked titles"

ratio_df["cohort"] = ratio_df["title_id"].apply(lambda t: "pre_wave" if t in PRE_WAVE else "post_wave")
cohort_stats = ratio_df.groupby("cohort")["ratio_pct"].agg(["mean", "median", "count"])
print(cohort_stats.round(2).to_string())

zero_by_cohort = ratio_df[ratio_df["ratio_pct"] == 0].groupby("cohort")["title_id"].count()
print(f"\ntitles at exactly 0%: pre_wave={zero_by_cohort.get('pre_wave', 0)}/13, "
      f"post_wave={zero_by_cohort.get('post_wave', 0)}/10")

           mean  median  count
cohort                        
post_wave  9.19    11.6     10
pre_wave   9.98     0.0     13

titles at exactly 0%: pre_wave=7/13, post_wave=4/10


**Honest read, not overclaimed**: pre-wave and post-wave means are close
(mean ~10% each) — but their MEDIANS diverge sharply (pre-wave's median is
0%, post-wave's is over 11%), because pre-wave titles split between a few
very high values (Counter-Strike, League of Legends, Age of Empires II all
above the whole-cohort norm) and many exact zeros, while post-wave titles
cluster more moderately across nonzero values. **This is not evidence of a
structural newer-vs-older difference** — this is a single-week snapshot, so
what it actually measures is "which titles happened to have a tournament
running (or tournament-tagged content circulating) during this specific
week," which is largely a timing artifact, not a deep age/era effect. A
real test of that question would need this same per-title ratio computed
over many weeks/months, not one.

## Present-day niche composition

Same 7 genre×platform niche cells as `notebooks/niche_stacked_area.ipynb`,
built the same way from `config/titles.yaml`. Composition here is each
title's share of its niche's total `own_collector` hours across the whole
window (not just the latest single poll, which would be far too noisy given
the sparse-polling caveat above).

In [8]:
config_titles = yaml.safe_load(open(REPO_ROOT / "config" / "titles.yaml"))["titles"]
niche_members = defaultdict(list)
for t in config_titles:
    niche_members[(t["genre"], t["platform"])].append(t["id"])
niche_cells = {k: v for k, v in niche_members.items() if len(v) >= 2}

hours_by_title = dict(conn.execute(
    "SELECT title_id, SUM(hours_watched) FROM monthly_category_history WHERE source = 'own_collector' "
    "GROUP BY title_id"
).fetchall())

rows = []
for (genre, platform), members in sorted(niche_cells.items()):
    vals = {t: hours_by_title.get(t, 0.0) for t in members}
    total = sum(vals.values())
    for t, v in sorted(vals.items(), key=lambda kv: -kv[1]):
        rows.append({"niche": f"{genre}/{platform}", "title_id": t,
                     "hours": v, "share_pct": round(100 * v / total, 1) if total else 0.0})

present_day_df = pd.DataFrame(rows)
for niche, group in present_day_df.groupby("niche", sort=False):
    print(f"\n{niche}:")
    print(group[["title_id", "hours", "share_pct"]].to_string(index=False))


battle_royale/mobile:
   title_id         hours  share_pct
pubg_mobile 194550.912500       87.2
  free_fire  28574.689722       12.8

battle_royale/pc_console:
    title_id        hours  share_pct
    fortnite 7.440036e+06       54.8
apex_legends 4.325388e+06       31.8
        pubg 1.815402e+06       13.4

fighting/console:
      title_id        hours  share_pct
street_fighter 1.477914e+06       79.9
        tekken 3.098026e+05       16.7
 mortal_kombat 6.192090e+04        3.3

moba/mobile:
         title_id         hours  share_pct
mobile_legends_bb 420203.543333       78.7
        wild_rift 113862.673056       21.3

moba/pc:
         title_id        hours  share_pct
league_of_legends 1.644758e+07       71.2
            dota2 6.636998e+06       28.8

rts/pc:
         title_id         hours  share_pct
age_of_empires_ii 325893.841389       65.1
       starcraft2 174428.848889       34.9

tac_fps/pc:
         title_id        hours  share_pct
   counter_strike 1.405510e+07       50.5
  

## Comparison against Kaggle's final available month (2024-09)

`esports_share_of_twitch.ipynb`'s headline figure for 2024-09 (its last
available month) was **29.2%** — tracked-title hours as a % of that month's
top-200-only total. Metric (a) above uses a broader, true whole-Twitch
denominator, so the mechanical expectation (all else equal) is that metric
(a) should sit at or BELOW 29.2%, not above it — a bigger denominator means
a smaller share, not a bigger one.

**Frame this explicitly as suggestive, not conclusive**: there are 23 months
of zero direct observation between the two points, one thin present-day
sample (see the `platform_totals` caveat above) is being compared against
one historical point, and the two use different-population denominators on
top of that. This is not a trend line — it's two dots with a large gap and
a methodological difference between them.

In [9]:
kaggle_final_month_share = 29.2  # esports_share_of_twitch.ipynb's 2024-09 figure, top-200-only denominator

print(f"Kaggle final month (2024-09), top-200-only denominator: {kaggle_final_month_share:.1f}%")
print(f"Present-day (this notebook), true whole-Twitch denominator: {metric_a:.1f}%")
print()
if metric_a > kaggle_final_month_share:
    print(f"Present-day sits ABOVE the Kaggle final-month figure ({metric_a:.1f}% vs. {kaggle_final_month_share:.1f}%), "
          "despite using a broader denominator that should mechanically push it lower, not higher. "
          "Given how thin the present-day platform_totals denominator is (see the caveat above), "
          "this divergence should be read as a sign the present-day estimate carries real uncertainty, "
          "not as evidence esports' share has actually risen since 2024-09 — the honest read is "
          "'not inconsistent with continued decline, not inconsistent with stabilization either,' "
          "not a confirmed reversal.")
else:
    print(f"Present-day sits at or below the Kaggle final-month figure ({metric_a:.1f}% vs. "
          f"{kaggle_final_month_share:.1f}%), consistent with the mechanical expectation from the broader "
          "denominator and with the declining trend esports_share_of_twitch.ipynb already found. Still "
          "only suggestive, per the caveats above — not a confirmed continuation of that trend.")

Kaggle final month (2024-09), top-200-only denominator: 29.2%
Present-day (this notebook), true whole-Twitch denominator: 33.9%

Present-day sits ABOVE the Kaggle final-month figure (33.9% vs. 29.2%), despite using a broader denominator that should mechanically push it lower, not higher. Given how thin the present-day platform_totals denominator is (see the caveat above), this divergence should be read as a sign the present-day estimate carries real uncertainty, not as evidence esports' share has actually risen since 2024-09 — the honest read is 'not inconsistent with continued decline, not inconsistent with stabilization either,' not a confirmed reversal.
